**SparkSession**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, sum as spark_sum, round

spark = SparkSession.builder \
    .appName("Tugas6-ETL") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


**A. EXTRACT**

In [3]:
# Membaca data transaksi dari CSV
df_transaksi = spark.read.csv(
    "tugas6_transaksi.csv",
    header=True,
    inferSchema=True
)

# Membaca data produk dari JSON Lines
df_produk = spark.read.json("tugas6_produk.json")

# Membaca data ulasan dari CSV
df_ulasan = spark.read.csv(
    "tugas6_ulasan.csv",
    header=True,
    inferSchema=True
)

print("Jumlah data transaksi :", df_transaksi.count())
print("Jumlah data produk    :", df_produk.count())
print("Jumlah data ulasan    :", df_ulasan.count())

Jumlah data transaksi : 5000
Jumlah data produk    : 30
Jumlah data ulasan    : 3500


Menampilkan schema masing-masing data

In [4]:
print("=== Schema Data Transaksi ===")
df_transaksi.printSchema()

print("=== Schema Data Produk ===")
df_produk.printSchema()

print("=== Schema Data Ulasan ===")
df_ulasan.printSchema()

=== Schema Data Transaksi ===
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

=== Schema Data Produk ===
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

=== Schema Data Ulasan ===
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



**B. TRANSFORM**

In [5]:
# Join transaksi dengan ulasan
df_etl = df_transaksi.join(
    df_ulasan,
    on="order_id",
    how="left"
)

# Join hasil sebelumnya dengan data produk
df_etl = df_etl.join(
    df_produk,
    on="product_id",
    how="inner"
)

df_etl.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|
+----------+--------+------------+-------------------+------+------+------------+-----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|
+----------+--------+------------+-------------------+------+------+------------+-----------+
only showing top 5 rows



Menambahkan kolom **total_pendapatan**

In [6]:
df_etl = df_etl.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga")
)

df_etl.select(
    "order_id",
    "product_id",
    "nama_produk",
    "kategori",
    "unit_terjual",
    "harga",
    "total_pendapatan",
    "rating"
).show(10)

+--------+----------+-----------+------------+------------+------+----------------+------+
|order_id|product_id|nama_produk|    kategori|unit_terjual| harga|total_pendapatan|rating|
+--------+----------+-----------+------------+------------+------+----------------+------+
|     TX0|        21|  Produk-21|  Elektronik|           2| 25000|           50000|     1|
|     TX1|         8|   Produk-8|     Fashion|           6|250000|         1500000|  NULL|
|     TX2|        25|  Produk-25|  Elektronik|           4| 25000|          100000|     2|
|     TX3|         3|   Produk-3|     Fashion|           4| 75000|          300000|     5|
|     TX4|        19|  Produk-19|Rumah Tangga|           6| 75000|          450000|  NULL|
|     TX5|        10|  Produk-10|  Elektronik|           3| 75000|          225000|     5|
|     TX6|        26|  Produk-26|   Kesehatan|           6| 75000|          450000|     4|
|     TX7|         4|   Produk-4|     Makanan|           1|150000|          150000|  NULL|

**C. TRANSFORM**

In [7]:
# Menentukan apakah transaksi memiliki ulasan
df_etl = df_etl.withColumn(
    "ada_ulasan",
    col("rating").isNotNull()
)

df_etl.select(
    "order_id",
    "rating",
    "ada_ulasan"
).show(10)

+--------+------+----------+
|order_id|rating|ada_ulasan|
+--------+------+----------+
|     TX0|     1|      true|
|     TX1|  NULL|     false|
|     TX2|     2|      true|
|     TX3|     5|      true|
|     TX4|  NULL|     false|
|     TX5|     5|      true|
|     TX6|     4|      true|
|     TX7|  NULL|     false|
|     TX8|     5|      true|
|     TX9|  NULL|     false|
+--------+------+----------+
only showing top 10 rows



Kemudian nilai rating yang masih **null** diisi dengan **0**.

In [8]:
# Mengisi rating kosong dengan 0
df_etl = df_etl.na.fill(
    {"rating": 0}
)

df_etl.select(
    "order_id",
    "rating",
    "ada_ulasan"
).show(10)

+--------+------+----------+
|order_id|rating|ada_ulasan|
+--------+------+----------+
|     TX0|     1|      true|
|     TX1|     0|     false|
|     TX2|     2|      true|
|     TX3|     5|      true|
|     TX4|     0|     false|
|     TX5|     5|      true|
|     TX6|     4|      true|
|     TX7|     0|     false|
|     TX8|     5|      true|
|     TX9|     0|     false|
+--------+------+----------+
only showing top 10 rows



Mengecek hasil akhir Transform

In [9]:
df_etl.select(
    "order_id",
    "product_id",
    "nama_produk",
    "kategori",
    "unit_terjual",
    "harga",
    "total_pendapatan",
    "rating",
    "ada_ulasan"
).show(10)

+--------+----------+-----------+------------+------------+------+----------------+------+----------+
|order_id|product_id|nama_produk|    kategori|unit_terjual| harga|total_pendapatan|rating|ada_ulasan|
+--------+----------+-----------+------------+------------+------+----------------+------+----------+
|     TX0|        21|  Produk-21|  Elektronik|           2| 25000|           50000|     1|      true|
|     TX1|         8|   Produk-8|     Fashion|           6|250000|         1500000|     0|     false|
|     TX2|        25|  Produk-25|  Elektronik|           4| 25000|          100000|     2|      true|
|     TX3|         3|   Produk-3|     Fashion|           4| 75000|          300000|     5|      true|
|     TX4|        19|  Produk-19|Rumah Tangga|           6| 75000|          450000|     0|     false|
|     TX5|        10|  Produk-10|  Elektronik|           3| 75000|          225000|     5|      true|
|     TX6|        26|  Produk-26|   Kesehatan|           6| 75000|          450000

**D. LOAD**

In [13]:
# Membuat folder tujuan di HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6

In [16]:
# Menyimpan hasil ETL ke HDFS dalam format Parquet
df_etl.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet(
        "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"
    )

print("Hasil ETL berhasil disimpan ke HDFS.")

[Stage 37:>                                                         (0 + 1) / 1]

Hasil ETL berhasil disimpan ke HDFS.


Verifikasi hasil penyimpanan

In [17]:
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

-rw-r--r--   3 aufaa supergroup          0 2026-10-01 00:00 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - aufaa supergroup          0 2026-10-01 00:00 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 aufaa supergroup      16366 2026-10-01 00:00 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-19394491-1a84-4922-842f-0f9f411528b4.c000.snappy.parquet
drwxr-xr-x   - aufaa supergroup          0 2026-10-01 00:00 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 aufaa supergroup      11133 2026-10-01 00:00 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-19394491-1a84-4922-842f-0f9f411528b4.c000.snappy.parquet
drwxr-xr-x   - aufaa supergroup          0 2026-10-01 00:00 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 aufaa supergroup      10392 2026-10-01 00:00 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-19394491-1a84-4922-842f-0f9f411528b4.c000.snappy.parquet
drwxr-xr-x   - aufaa sup

Membaca kembali data dari HDFS

In [18]:
df_hasil = spark.read.parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"
)

print("Jumlah baris hasil ETL:", df_hasil.count())

Jumlah baris hasil ETL: 5000


**E. INSIGHT AKHIR**

In [19]:
df_insight = df_hasil.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    spark_sum(
        when(col("ada_ulasan") == True, 1).otherwise(0)
    ).alias("transaksi_dengan_ulasan")
)

df_insight.show()

+------------+---------------+-----------------------+
|    kategori|total_transaksi|transaksi_dengan_ulasan|
+------------+---------------+-----------------------+
|  Elektronik|           1653|                   1168|
|     Fashion|           1035|                    726|
|   Kesehatan|            961|                    662|
|Rumah Tangga|            815|                    575|
|     Makanan|            536|                    369|
+------------+---------------+-----------------------+



Kemudian hitung persentase transaksi yang memiliki ulasan.

In [20]:
df_insight = df_insight.withColumn(
    "persentase_ulasan",
    round(
        col("transaksi_dengan_ulasan") /
        col("total_transaksi") * 100,
        2
    )
)

df_insight.orderBy("persentase_ulasan").show()

+------------+---------------+-----------------------+-----------------+
|    kategori|total_transaksi|transaksi_dengan_ulasan|persentase_ulasan|
+------------+---------------+-----------------------+-----------------+
|     Makanan|            536|                    369|            68.84|
|   Kesehatan|            961|                    662|            68.89|
|     Fashion|           1035|                    726|            70.14|
|Rumah Tangga|            815|                    575|            70.55|
|  Elektronik|           1653|                   1168|            70.66|
+------------+---------------+-----------------------+-----------------+



Menampilkan kategori dengan persentase ulasan paling rendah

In [21]:
kategori_terendah = df_insight.orderBy(
    col("persentase_ulasan").asc()
).first()

print("Kategori dengan persentase ulasan paling rendah:")
print("Kategori :", kategori_terendah["kategori"])
print("Total transaksi :", kategori_terendah["total_transaksi"])
print("Transaksi dengan ulasan :", kategori_terendah["transaksi_dengan_ulasan"])
print("Persentase ulasan :", kategori_terendah["persentase_ulasan"], "%")

Kategori dengan persentase ulasan paling rendah:
Kategori : Makanan
Total transaksi : 536
Transaksi dengan ulasan : 369
Persentase ulasan : 68.84 %


Penutup SparkSession

In [22]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
